In [35]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [4]:
dataset = pd.read_csv('portugal_listings.csv')

C:\Users\PC\AppData\Local\Temp\ipykernel_20784\1801581679.py:1: DtypeWarning: Columns (9,11,12,13,14,15,19) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv('portugal_listings.csv')


In [5]:
print(dataset.shape)
print(dataset.head())

(174727, 23)
      Price   District                  City  \
0  780000.0  Vila Real              Valpaços   
1  228000.0       Faro  São Brás de Alportel   
2  250000.0       Faro  São Brás de Alportel   
3  250000.0       Faro  São Brás de Alportel   
4  158000.0       Faro              Portimão   

                               Town       Type EnergyCertificate  TotalArea  \
0  Carrazedo de Montenegro e Curros       Farm                NC   552450.0   
1              São Brás de Alportel  Apartment                A+      108.0   
2              São Brás de Alportel  Apartment                A+      114.0   
3              São Brás de Alportel  Apartment                A+      114.0   
4                          Portimão  Apartment                 D    21953.0   

   NumberOfBathrooms  Parking         Floor  ...  Garage Elevator  \
0                0.0      0.0           NaN  ...     NaN    False   
1                2.0      1.0  Ground Floor  ...     NaN     True   
2               

In [6]:
print(dataset.isna().sum())

Price                       419
District                      0
City                          0
Town                          2
Type                         16
EnergyCertificate            14
TotalArea                 14450
NumberOfBathrooms         15762
Parking                      32
Floor                    146798
ConstructionYear          60467
EnergyEfficiencyLevel     68247
PublishDate              107656
Garage                    68247
Elevator                     32
ElectricCarsCharging      68247
TotalRooms                88263
NumberOfBedrooms          99904
NumberOfWC               102355
ConservationStatus       145905
LivingArea                39480
LotSize                  115244
BuiltArea                104851
dtype: int64


In [7]:
# Since the target variable "Price" has a small number of missing values, those rows will be dropped.
dataset = dataset.loc[dataset['Price'].notna()]

In [8]:
print(dataset['Price'].isna().sum())
print(dataset.shape)

0
(174308, 23)


In [9]:
# As for the other missing values, we will consider them below.
# Since Town, Type, EnergyCertificate, Parking and Elevator each have less than 50 missing values, we will drop those rows and not much information will be lost.
dataset = dataset.loc[dataset[['Town', 'Type', 'EnergyCertificate', 'Parking', 'Elevator']].notna().all(axis=1)]

In [10]:
print(dataset.isna().sum())

Price                         0
District                      0
City                          0
Town                          0
Type                          0
EnergyCertificate             0
TotalArea                 14407
NumberOfBathrooms         15715
Parking                       0
Floor                    146428
ConstructionYear          60237
EnergyEfficiencyLevel     68072
PublishDate              107325
Garage                    68072
Elevator                      0
ElectricCarsCharging      68072
TotalRooms                88054
NumberOfBedrooms          99686
NumberOfWC               102109
ConservationStatus       145529
LivingArea                39407
LotSize                  114918
BuiltArea                104513
dtype: int64


In [11]:
# Columns TotalArea and NumberOfBathrooms have 8-9% missing values.
# Since this is not such a low percentage, we are going to fill those values instead of dropping rows. The question is - with what, median or average (mean) value?
# We will use median, not mean, because real estate data usually contains outliers (large villas vs small apartments).

# Before filling, we are going to group rows by a related column (e.g. property Type) instead of using one global median for the whole dataset.
# Why? Because a studio apartment and a villa have very different typical areas - a single overall median would not make sense for either one.
# Grouping lets us fill each row with a median that's realistic for that specific type of property.

In [12]:
groupsByArea = dataset.groupby('Type')['TotalArea']
mediansByArea = groupsByArea.transform('median')
dataset['TotalArea'] = dataset['TotalArea'].fillna(mediansByArea)

In [13]:
groupsByBathrooms = dataset.groupby('Type')['NumberOfBathrooms']
mediansByBathrooms = groupsByBathrooms.transform('median')
dataset['NumberOfBathrooms'] = dataset['NumberOfBathrooms'].fillna(mediansByBathrooms)

In [14]:
print(dataset['TotalArea'].isna().sum())
print(dataset['NumberOfBathrooms'].isna().sum())

0
0


In [15]:
# Before further filling, we will drop the columns BuiltArea and NumberOfWC since they are similar to TotalArea and NumberOfBathrooms, but have 100,000+ missing values.
# Using the same logic, we will drop NumberOfBedrooms - we already have TotalRooms and NumberOfBathrooms, so it can easily be estimated from those.
# We will also drop the column PublishDate, because the date an ad was posted is not relevant to price prediction.

In [16]:
dataset = dataset.drop(columns=['BuiltArea', 'NumberOfWC', 'PublishDate', 'NumberOfBedrooms'])
print(dataset.isna().sum())

Price                         0
District                      0
City                          0
Town                          0
Type                          0
EnergyCertificate             0
TotalArea                     0
NumberOfBathrooms             0
Parking                       0
Floor                    146428
ConstructionYear          60237
EnergyEfficiencyLevel     68072
Garage                    68072
Elevator                      0
ElectricCarsCharging      68072
TotalRooms                88054
ConservationStatus       145529
LivingArea                39407
LotSize                  114918
dtype: int64


In [17]:
# Dealing with boolean columns (Garage, ElectricCarsCharging)

dataset['Garage'].unique()
dataset['ElectricCarsCharging'].unique()

array([nan, False, True], dtype=object)

In [18]:
# Here we have two options: we can treat the nan value as unknown, or we can assume that nan is closer to False.
# We will go with the second option, since in real estate listings, sellers almost always mention a garage or EV charging when the property has one.

dataset['Garage'] = dataset['Garage'].fillna(False)
dataset['ElectricCarsCharging'] = dataset['ElectricCarsCharging'].fillna(False)

C:\Users\PC\AppData\Local\Temp\ipykernel_20784\2625979054.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset['Garage'] = dataset['Garage'].fillna(False)
C:\Users\PC\AppData\Local\Temp\ipykernel_20784\2625979054.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset['ElectricCarsCharging'] = dataset['ElectricCarsCharging'].fillna(False)


In [19]:
print(dataset.isna().sum())

Price                         0
District                      0
City                          0
Town                          0
Type                          0
EnergyCertificate             0
TotalArea                     0
NumberOfBathrooms             0
Parking                       0
Floor                    146428
ConstructionYear          60237
EnergyEfficiencyLevel     68072
Garage                        0
Elevator                      0
ElectricCarsCharging          0
TotalRooms                88054
ConservationStatus       145529
LivingArea                39407
LotSize                  114918
dtype: int64


In [20]:
# For columns with not too many missing values, we will again use the median.

groupsConstructionYear = dataset.groupby('Type')['ConstructionYear']
mediansConstructionYear = groupsConstructionYear.transform('median')
dataset['ConstructionYear'] = dataset['ConstructionYear'].fillna(mediansConstructionYear)

groupsTotalRooms = dataset.groupby('Type')['TotalRooms']
mediansTotalRooms = groupsTotalRooms.transform('median')
dataset['TotalRooms'] = dataset['TotalRooms'].fillna(mediansTotalRooms)

groupsLivingArea = dataset.groupby('Type')['LivingArea']
mediansLivingArea = groupsLivingArea.transform('median')
dataset['LivingArea'] = dataset['LivingArea'].fillna(mediansLivingArea)

In [21]:
# For EnergyEfficiencyLevel, which has categorical values, we cannot use median. Here, we will use mode (the most common value) instead.
# Since mode can return more than one value, the syntax is a bit different.

groupsEnergy = dataset.groupby('Type')['EnergyEfficiencyLevel']
modes = groupsEnergy.transform(lambda x: x.mode()[0] if not x.mode().empty else 'Unknown')
dataset['EnergyEfficiencyLevel'] = dataset['EnergyEfficiencyLevel'].fillna(modes)

In [22]:
# Finally, we are dropping the columns with 100,000+ missing values. These columns would have been useful, but with such a large number of missing values, filling them in would 
# introduce too much uncertainty to be reliable for the model.

dataset = dataset.drop(columns=['Floor', 'ConservationStatus', 'LotSize'])

In [23]:
print(dataset.isna().sum())

Price                    0
District                 0
City                     0
Town                     0
Type                     0
EnergyCertificate        0
TotalArea                0
NumberOfBathrooms        0
Parking                  0
ConstructionYear         0
EnergyEfficiencyLevel    0
Garage                   0
Elevator                 0
ElectricCarsCharging     0
TotalRooms               0
LivingArea               0
dtype: int64


In [24]:
print(dataset.shape)
print(dataset.head())

(174272, 16)
      Price   District                  City  \
0  780000.0  Vila Real              Valpaços   
1  228000.0       Faro  São Brás de Alportel   
2  250000.0       Faro  São Brás de Alportel   
3  250000.0       Faro  São Brás de Alportel   
4  158000.0       Faro              Portimão   

                               Town       Type EnergyCertificate  TotalArea  \
0  Carrazedo de Montenegro e Curros       Farm                NC   552450.0   
1              São Brás de Alportel  Apartment                A+      108.0   
2              São Brás de Alportel  Apartment                A+      114.0   
3              São Brás de Alportel  Apartment                A+      114.0   
4                          Portimão  Apartment                 D    21953.0   

   NumberOfBathrooms  Parking  ConstructionYear EnergyEfficiencyLevel  Garage  \
0                0.0      0.0            1970.0                    NC   False   
1                2.0      1.0            2000.0              

In [25]:
dataset.describe()

,Price,TotalArea,NumberOfBathrooms,Parking,ConstructionYear,TotalRooms,LivingArea
count,1.742720e+05,1.742720e+05,174272.000000,174272.000000,174272.000000,174272.000000,1.742720e+05
mean,3.869344e+05,3.756238e+05,1.552091,0.501348,1990.706092,2.629160,1.156136e+03
std,3.367514e+06,1.471699e+08,7.434640,0.776916,22.144461,7.051074,2.757508e+04
min,1.000000e+00,-7.196067e+06,-13.000000,0.000000,1900.000000,0.000000,0.000000e+00
25%,9.000000e+04,9.200000e+01,0.000000,0.000000,1986.000000,1.000000,8.700000e+01
50%,2.300000e+05,1.640000e+02,1.000000,0.000000,1990.000000,3.000000,1.380000e+02
75%,4.200000e+05,6.740000e+02,2.000000,1.000000,2002.000000,4.000000,4.000000e+02
max,1.380000e+09,6.142007e+10,3020.000000,3.000000,2026.000000,2751.000000,5.429000e+06


In [26]:
dataset.describe(include='object')

,District,City,Town,Type,EnergyCertificate,EnergyEfficiencyLevel,Elevator
count,174272,174272,174272,174272,174272,174272,174272
unique,28,280,2329,21,11,11,2
top,Lisboa,Lisboa,Paranhos,Apartment,NC,NC,False
freq,41338,11197,2529,62619,77146,111897,130779


In [27]:
# Here, we see some impossible values. For TotalArea and NumberOfBathrooms, the min is negative and the max is massive. For TotalRooms, the max is 2751, and for LivingArea it's 5.4 million m².
# The Price max is potentially too large as well.
# These are not outliers, they are data errors.

In [28]:
# Removing physically impossible values

dataset = dataset[dataset['TotalArea'] > 0]
dataset = dataset[dataset['NumberOfBathrooms'] >= 0]
dataset = dataset[dataset['TotalRooms'] >= 0]
dataset = dataset[dataset['LivingArea'] >= 0]
dataset = dataset[dataset['Price'] > 0]

In [29]:
# Using IQR method to set an upper limit
# The multiplier is intentionally set to a larger value than usual, since we do not want to delete legitimate large real estate listings (e.g. villas).

def remove_outliers_iqr(df, column, multiplier=5):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + multiplier * IQR
    return df[df[column] <= upper_bound]

dataset = remove_outliers_iqr(dataset, 'TotalArea')
dataset = remove_outliers_iqr(dataset, 'Price')
dataset = remove_outliers_iqr(dataset, 'LivingArea')
dataset = remove_outliers_iqr(dataset, 'TotalRooms')
dataset = remove_outliers_iqr(dataset, 'NumberOfBathrooms')

In [30]:
dataset.describe()

,Price,TotalArea,NumberOfBathrooms,Parking,ConstructionYear,TotalRooms,LivingArea
count,1.552840e+05,155284.000000,155284.000000,155284.000000,155284.000000,155284.000000,155284.000000
mean,3.178735e+05,405.781851,1.568256,0.506066,1991.193771,2.668350,315.938084
std,3.172680e+05,583.439535,1.438026,0.753669,22.303849,2.087364,403.793854
min,1.000000e+00,1.000000,0.000000,0.000000,1900.000000,0.000000,0.000000
25%,9.800000e+04,88.000000,0.000000,0.000000,1986.000000,1.000000,84.000000
50%,2.350000e+05,143.000000,1.000000,0.000000,1991.000000,3.000000,124.000000
75%,4.000000e+05,360.000000,2.000000,1.000000,2003.000000,4.000000,275.000000
max,2.025000e+06,3644.000000,12.000000,3.000000,2026.000000,19.000000,1365.000000


In [ ]:
# Columns that are not numeric (District, City, Town, Type, EnergyCertificate, EnergyEfficiencyLevel) must be transformed.

In [34]:
print(dataset[['District', 'City', 'Town', 'Type', 'EnergyCertificate', 'EnergyEfficiencyLevel']].nunique())

District                   28
City                      280
Town                     2314
Type                       21
EnergyCertificate          11
EnergyEfficiencyLevel      11
dtype: int64


In [ ]:
# Standard One-Hot Encoding (OHE) would not work well here. OHE creates a new binary column for every unique value, which works well for columns with a small number of values.
# Here, it would work for District, Type, EnergyCertificate, and EnergyEfficiencyLevel, and maybe even for City.
# But Town has too many different values. OHE would create 2314 new columns. We will use target encoding instead, replacing each category with the average price for that category.

In [ ]:
# First we do a train/test split, then target encoding for Town, and finally OHE for the rest of the columns.

X = dataset.drop(columns=['Price'])
y = dataset['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# We are doing target encoding for Town using only the train set, to avoid data leakage.

town_price_map = X_train.join(y_train).groupby('Town')['Price'].mean()

X_train['Town'] = X_train['Town'].map(town_price_map)
X_test['Town'] = X_test['Town'].map(town_price_map)

In [ ]:
# Potential problem here - if a town exists in the test set but not in the train set, map() will return NaN.

print(X_test['Town'].isna().sum())

24


In [ ]:
# We will handle this problem by using the global mean price from the train set.

global_mean_price = y_train.mean()
X_test['Town'] = X_test['Town'].fillna(global_mean_price)

In [41]:
# OHE

categorical_cols = ['District', 'City', 'Type', 'EnergyCertificate', 'EnergyEfficiencyLevel']

X_train = pd.get_dummies(X_train, columns=categorical_cols)
X_test = pd.get_dummies(X_test, columns=categorical_cols)

In [ ]:
# Potential problem here - if some value exists in the train set, but not in the test set (or vice versa), get_dummies() will create a different number of columns.

print(X_train.shape)
print(X_test.shape)

(124227, 361)
(31057, 344)


In [ ]:
# We will fix this by aligning the columns, using the train set as the reference since it has more columns.

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [44]:
print(X_train.shape)
print(X_test.shape)

(124227, 361)
(31057, 361)


In [ ]:
# The first model we will use for predictions is Linear Regression.